In [1]:
# ==============================================================================
# Step 0: Setup libraries, connection, and project parameters
# ==============================================================================
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import wrds

# Connect to the WRDS database
conn = wrds.Connection()

# Settings for Project 2: Industry, Event Window, and Rule Parameters
SIC         = "61"            # Target industry: Nondepository Credit Institutions
EVENT_START = "2021-01-01"    # Rule 2: Announcement start date (rdq >= 2021-01-01)
EVENT_END   = "2024-12-31"    # Rule 2: Announcement end date (rdq <= 2024-12-31)
PULL_START  = "2019-01-01"    # Historical start date for 4-quarter lag EPS and price
PULL_END    = "2024-12-31"    # Historical accounting end date
RET_START   = "2020-12-01"    # Daily returns start date (covers 3-day window for early 2021)
RET_END     = "2025-01-31"    # Daily returns end date (covers 3-day window for late 2024)
MIN_PRC     = 5               # Rule 4: Drop if last quarter price < $5
MAX_SUE     = 1               # Rule 5: Drop if |SUE| > 1
CLIENT_TICKER = "AXP"           # Target client ticker symbol

Loading library list...
Done


In [2]:
# ==============================================================================
# Step 1: Load Link table and filter by industry (SIC 61)
# ==============================================================================
# Read the link file provided by the instructor
link_raw = pd.read_csv("ccm_link.csv", dtype={"gvkey": str})

# Ensure sic2 is treated as string for consistent matching
link_raw["sic2"] = link_raw["sic2"].astype(str)

# Filter for the assigned industry: SIC 61
link = link_raw[link_raw["sic2"] == SIC].copy()

# Convert link date columns to datetime objects
link["linkdt"] = pd.to_datetime(link["linkdt"])
link["linkenddt"] = pd.to_datetime(link["linkenddt"].fillna("2099-12-31"))

# Create a quoted, comma-separated string of gvkeys for the SQL query
gv_list = "'" + "','".join(link["gvkey"].unique()) + "'"

print("Number of links in SIC 61:", len(link))
print("Number of unique companies (gvkey):", link["gvkey"].nunique())

# Look up client gvkey automatically using ticker
CLIENT_GVKEY = link_raw.loc[link_raw["ticker"] == CLIENT_TICKER, "gvkey"].iloc[0]
print(f"Client: {CLIENT_TICKER} -> gvkey: {CLIENT_GVKEY}")

Number of links in SIC 61: 52
Number of unique companies (gvkey): 47
Client: AXP -> gvkey: 001447


In [3]:
# ==============================================================================
# Step 4: Pull quarterly financial data from Compustat (comp.fundq)
# ==============================================================================
# SQL query to extract required quarterly columns for SIC 61 firms
fq = conn.raw_sql(f"""
    select gvkey, datadate, fyearq, fqtr, rdq, epspxq, prccq
    from comp.fundq
    where gvkey in ({gv_list})
      and datadate between '{PULL_START}' and '{PULL_END}'
      and indfmt = 'INDL' 
      and datafmt = 'STD' 
      and popsrc = 'D' 
      and consol = 'C'
""", date_cols=["datadate", "rdq"])

# Clean types: ensure gvkey is string and numeric fields are standard floats
fq["gvkey"] = fq["gvkey"].astype(str)
fq["epspxq"] = fq["epspxq"].astype(float)
fq["prccq"] = fq["prccq"].astype(float)

print("Step 4 Raw Compustat rows pulled:", len(fq))
print("Step 4 Unique companies pulled:", fq["gvkey"].nunique())
fq.head()

Step 4 Raw Compustat rows pulled: 1051
Step 4 Unique companies pulled: 46


,gvkey,datadate,fyearq,fqtr,rdq,epspxq,prccq
0,001447,2019-03-31,2019,1,2019-04-18,1.81,109.30
1,003592,2019-03-31,2019,1,2019-04-29,1.12,31.75
2,010121,2019-03-31,2019,1,2019-04-23,0.52,11.57
3,011925,2019-03-31,2019,1,2019-05-01,-0.27,2.87
4,015153,2019-03-31,2019,1,2019-05-02,2.05,72.43


In [4]:
# ==============================================================================
# Step 5: Merge 1 (Link table), Merge 2 (Lagged EPS & Price), and Compute SUE
# ==============================================================================

# --- Step 5a: Merge 1 (Link to CRSP permno) ---
# Match Compustat observations with CRSP identifiers using gvkey
panel = fq.merge(link[["gvkey", "permno", "linkdt", "linkenddt"]], on="gvkey", how="inner")

# Apply link date validity rule: linkdt <= rdq <= linkenddt
valid_link = (panel["linkdt"] <= panel["rdq"]) & (panel["rdq"] <= panel["linkenddt"])
panel = panel[valid_link].copy().reset_index(drop=True)

# Drop any duplicate rows for the same firm and fiscal quarter
panel = panel.drop_duplicates(subset=["gvkey", "fyearq", "fqtr"]).reset_index(drop=True)
panel = panel.drop(columns=["linkdt", "linkenddt"])

print("After Merge 1 (valid link matched):", len(panel), "rows")

# --- Step 5b: Construct continuous quarter index ---
# Example: 2021 Q1 -> 2021 * 4 + 1 = 8085
panel["qtr_id"] = (panel["fyearq"] * 4 + panel["fqtr"]).astype(int)

# --- Step 5c: Merge 2 (Self-merge for 4-quarter lag EPS and 1-quarter lag Price) ---
# 1. Match EPS from 4 quarters ago (shift qtr_id by +4 so it meets the current quarter)
lag4 = panel[["gvkey", "qtr_id", "epspxq"]].copy()
lag4["qtr_id"] = lag4["qtr_id"] + 4
lag4 = lag4.rename(columns={"epspxq": "eps_lag4"})
panel = panel.merge(lag4, on=["gvkey", "qtr_id"], how="left")

# 2. Match Price from 1 quarter ago (shift qtr_id by +1 so it meets the current quarter)
lag1 = panel[["gvkey", "qtr_id", "prccq"]].copy()
lag1["qtr_id"] = lag1["qtr_id"] + 1
lag1 = lag1.rename(columns={"prccq": "prc_lag1"})
panel = panel.merge(lag1, on=["gvkey", "qtr_id"], how="left")

# --- Step 5d: Compute SUE (Standardized Unexpected Earnings) ---
# Formula: (Current EPS - EPS 4 quarters ago) / Price 1 quarter ago
panel["sue"] = (panel["epspxq"] - panel["eps_lag4"]) / panel["prc_lag1"]

print("After Merge 2 (total rows):", len(panel))
print("Rows with successfully computed SUE:", panel["sue"].notna().sum())

# Preview key columns
cols_preview = ["gvkey", "permno", "fyearq", "fqtr", "rdq", "epspxq", "eps_lag4", "prc_lag1", "sue"]
panel[cols_preview].head()

After Merge 1 (valid link matched): 958 rows
After Merge 2 (total rows): 958
Rows with successfully computed SUE: 771


,gvkey,permno,fyearq,fqtr,rdq,epspxq,eps_lag4,prc_lag1,sue
0,001447,59176,2019,1,2019-04-18,1.81,NaN,NaN,NaN
1,003592,14229,2019,1,2019-04-29,1.12,NaN,NaN,NaN
2,015153,80169,2019,1,2019-05-02,2.05,NaN,NaN,NaN
3,017180,11593,2019,1,2019-05-15,0.11,NaN,NaN,NaN
4,017306,13908,2019,1,2019-05-02,0.59,NaN,NaN,NaN


In [5]:
# ==============================================================================
# Step 6: Apply Rules 1 to 5 sequentially and record sample reduction counts
# ==============================================================================

# Dictionary to store sample retention counts for Check 1
sample_counts = {}
sample_counts["Initial Compustat rows pulled (Step 4)"] = len(fq)
sample_counts["After Merge 1 (valid CRSP link)"] = len(panel)

# Rule 1: Announcement date (rdq) must not be missing
df_r1 = panel[panel["rdq"].notna()].copy()
sample_counts["Rule 1: Drop if rdq is missing"] = len(df_r1)

# Rule 2: Announcement date must fall within the event window (2021-01-01 to 2024-12-31)
df_r2 = df_r1[(df_r1["rdq"] >= EVENT_START) & (df_r1["rdq"] <= EVENT_END)].copy()
sample_counts["Rule 2: Drop if rdq outside 2021-2024"] = len(df_r2)

# Rule 3: Must have valid SUE (non-missing earnings and 4-quarter lag EPS)
df_r3 = df_r2[df_r2["sue"].notna()].copy()
sample_counts["Rule 3: Drop if SUE is missing"] = len(df_r3)

# Rule 4: Drop if last quarter price < $5 (penny stocks filter)
df_r4 = df_r3[df_r3["prc_lag1"] >= MIN_PRC].copy()
sample_counts["Rule 4: Drop if prc_lag1 < $5"] = len(df_r4)

# Rule 5: Drop if extreme SUE (|SUE| > 1)
df_r5 = df_r4[df_r4["sue"].abs() <= MAX_SUE].copy().reset_index(drop=True)
sample_counts["Rule 5: Drop if |SUE| > 1"] = len(df_r5)

# Keep df_r5 as earnings_kept for daily returns extraction in Step 7
earnings_kept = df_r5.copy()

# Construct the Check 1 Sample Construction Table
sample_table = pd.DataFrame(list(sample_counts.items()), columns=["Step / Filter Description", "Remaining Rows"])
sample_table["Rows Dropped"] = sample_table["Remaining Rows"].shift(1) - sample_table["Remaining Rows"]
sample_table["Rows Dropped"] = sample_table["Rows Dropped"].fillna(0).astype(int)

print(f"Sample size after Rule 5: {len(earnings_kept)} announcements across {earnings_kept['permno'].nunique()} firms.")
sample_table

Sample size after Rule 5: 535 announcements across 39 firms.


,Step / Filter Description,Remaining Rows,Rows Dropped
0,Initial Compustat rows pulled (Step 4),1051,0
1,After Merge 1 (valid CRSP link),958,93
2,Rule 1: Drop if rdq is missing,958,0
3,Rule 2: Drop if rdq outside 2021-2024,656,302
4,Rule 3: Drop if SUE is missing,624,32
5,Rule 4: Drop if prc_lag1 < $5,535,89
6,Rule 5: Drop if |SUE| > 1,535,0


In [6]:
# ==============================================================================
# Step 7: Pull daily stock (crsp.dsf) and market (crsp.dsi) returns & Merge 3
# ==============================================================================

# 7a. Get unique permnos for companies kept after Rule 5
permno_list = "'" + "','".join(earnings_kept["permno"].astype(str).unique()) + "'"

# 7b. Pull daily individual stock returns from crsp.dsf
dsf = conn.raw_sql(f"""
    select permno, date, ret
    from crsp.dsf
    where permno in ({permno_list})
      and date between '{RET_START}' and '{RET_END}'
""", date_cols=["date"])

dsf["permno"] = dsf["permno"].astype(int)
dsf["ret"] = dsf["ret"].astype(float)

# 7c. Pull daily market returns from crsp.dsi and assign trading day index (tday)
dsi = conn.raw_sql(f"""
    select date, vwretd
    from crsp.dsi
    where date between '{RET_START}' and '{RET_END}'
""", date_cols=["date"])

dsi["vwretd"] = dsi["vwretd"].astype(float)
dsi = dsi.sort_values("date").reset_index(drop=True)
dsi["tday"] = dsi.index + 1  # 1, 2, 3... Sequential trading day counter

# 7d. Merge 3: Combine daily stock returns with market index and compute abnormal returns (ar)
daily = dsf.merge(dsi[["date", "vwretd", "tday"]], on="date", how="left")
daily["ar"] = daily["ret"] - daily["vwretd"]

print(f"Step 7 Complete: {len(daily)} stock-day observations pulled across {len(dsi)} trading days.")
daily.head()

Step 7 Complete: 39456 stock-day observations pulled across 1027 trading days.


,permno,date,ret,vwretd,tday,ar
0,11593,2020-12-01,-0.007491,0.009645,1,-0.017136
1,11593,2020-12-02,0.002516,0.001192,2,0.001324
2,11593,2020-12-03,-0.001255,0.002272,3,-0.003527
3,11593,2020-12-04,0.031407,0.010477,4,0.020930
4,11593,2020-12-07,-0.010962,-0.000220,5,-0.010742


In [7]:
# ==============================================================================
# Step 8: Match Day 0, extract 3-day window [-1, +1], compute CAR, apply Rule 6
# ==============================================================================

# 8a. Find Day 0 for each announcement (first trading day on or after rdq)
trading_days = dsi[["date", "tday"]].copy()

# Sort announcements by rdq and find the first available trading day
events = earnings_kept.sort_values("rdq").copy()
events = pd.merge_asof(
    events,
    trading_days,
    left_on="rdq",
    right_on="date",
    direction="forward"
)

# Rename trading day columns to clarify Day 0
events = events.rename(columns={"date": "day0_date", "tday": "day0_tday"})

# 8b. Construct 3-day event window: day -1 (tday - 1), day 0 (tday), day +1 (tday + 1)
# Create a temporary table expanding each announcement into 3 relative days
window_list = []
for rel_day in [-1, 0, 1]:
    temp = events[["gvkey", "permno", "fyearq", "fqtr", "rdq", "day0_tday", "sue"]].copy()
    temp["rel_day"] = rel_day
    temp["target_tday"] = temp["day0_tday"] + rel_day
    window_list.append(temp)

event_window = pd.concat(window_list, ignore_index=True)

# 8c. Merge daily abnormal returns onto the 3-day window
event_returns = event_window.merge(
    daily[["permno", "tday", "ret", "vwretd", "ar"]],
    left_on=["permno", "target_tday"],
    right_on=["permno", "tday"],
    how="inner"
)

# 8d. Aggregate 3-day CAR and count available trading days per announcement
car_summary = event_returns.groupby(
    ["gvkey", "permno", "fyearq", "fqtr", "rdq", "sue"], 
    as_index=False
).agg(
    n_days=("ar", "count"),
    car=("ar", "sum")
)

# 8e. Rule 6: Drop announcements with missing daily returns (must have exactly 3 days)
final_sample = car_summary[car_summary["n_days"] == 3].copy().reset_index(drop=True)

print(f"Announcements entering Step 8: {len(events)}")
print(f"Rule 6 Kept (Full 3-day window available): {len(final_sample)}")
print(f"Rule 6 Dropped: {len(events) - len(final_sample)}")
final_sample.head()

Announcements entering Step 8: 535
Rule 6 Kept (Full 3-day window available): 535
Rule 6 Dropped: 0


,gvkey,permno,fyearq,fqtr,rdq,sue,n_days,car
0,001447,59176,2020,4,2021-01-26,-0.002793,3,-0.073307
1,001447,59176,2021,1,2021-04-23,0.019271,3,0.013107
2,001447,59176,2021,2,2021-07-23,0.017817,3,-0.010132
3,001447,59176,2021,3,2021-10-22,0.005810,3,0.010066
4,001447,59176,2021,4,2022-01-25,0.002567,3,0.115009


In [8]:
# ==============================================================================
# Check 1: Final Sample Construction Table (Rules 1 to 6)
# ==============================================================================
# Append Rule 6 counts to the sample table
r6 = len(final_sample)
c6 = final_sample["gvkey"].nunique()

new_row = pd.DataFrame([{
    "Step / Filter Description": "Rule 6: Drop if 3-day return window incomplete",
    "Remaining Rows": r6,
    "Rows Dropped": len(earnings_kept) - r6
}])

final_sample_table = pd.concat([sample_table, new_row], ignore_index=True)

print("=" * 65)
print("CHECK 1: FINAL SAMPLE CONSTRUCTION TABLE")
print("=" * 65)
final_sample_table

CHECK 1: FINAL SAMPLE CONSTRUCTION TABLE


,Step / Filter Description,Remaining Rows,Rows Dropped
0,Initial Compustat rows pulled (Step 4),1051,0
1,After Merge 1 (valid CRSP link),958,93
2,Rule 1: Drop if rdq is missing,958,0
3,Rule 2: Drop if rdq outside 2021-2024,656,302
4,Rule 3: Drop if SUE is missing,624,32
5,Rule 4: Drop if prc_lag1 < $5,535,89
6,Rule 5: Drop if |SUE| > 1,535,0
7,Rule 6: Drop if 3-day return window incomplete,535,0


In [9]:
# Check 2: Duplicates
dup_count = final_sample.duplicated(subset=["gvkey", "rdq"]).sum()
print("Number of duplicate (gvkey, rdq) pairs in final sample:", dup_count)

Number of duplicate (gvkey, rdq) pairs in final sample: 0


In [10]:
# Check 3 (a): Announcement date after quarter-end date
check3_a = final_sample.merge(
    fq[["gvkey", "fyearq", "fqtr", "datadate"]],
    on=["gvkey", "fyearq", "fqtr"],
    how="left"
)
earlier_rdq = (check3_a["rdq"] <= check3_a["datadate"]).sum()

# Check 3 (b): Show day 0 is always an active trading day in dsi
trading_dates = set(dsi["date"].dt.strftime("%Y-%m-%d"))
day0_is_trading = events["day0_date"].dt.strftime("%Y-%m-%d").isin(trading_dates).all()

print("Announcements on or before quarter-end (datadate):", earlier_rdq)
print("Are all Day 0 dates valid trading days?", day0_is_trading)

Announcements on or before quarter-end (datadate): 0
Are all Day 0 dates valid trading days? True


In [11]:
# Check 4: Five largest absolute CAR announcements
top5_car_idx = final_sample["car"].abs().nlargest(5).index
cols_car = ["gvkey", "permno", "fyearq", "fqtr", "rdq", "sue", "car"]

check4_car = final_sample.loc[top5_car_idx, cols_car].copy().reset_index(drop=True)
check4_car["car_pct"] = (check4_car["car"] * 100).round(2).astype(str) + "%"
check4_car

,gvkey,permno,fyearq,fqtr,rdq,sue,car,car_pct
0,019647,15092,2021,2,2021-07-28,0.058717,0.467976,46.8%
1,065365,85705,2023,2,2023-11-14,0.042000,0.424242,42.42%
2,184392,11664,2023,3,2023-11-09,-0.011206,-0.365015,-36.5%
3,150477,89574,2023,1,2023-05-08,-0.073416,-0.317414,-31.74%
4,025852,78018,2022,3,2022-11-09,0.061463,0.308583,30.86%


In [12]:
# Check 5: Look up American Express (gvkey: 001447) 2021 Q1
axp_check = final_sample[
    (final_sample["gvkey"] == CLIENT_GVKEY) & 
    (final_sample["fyearq"] == 2021) & 
    (final_sample["fqtr"] == 1)
]
axp_check[["gvkey", "permno", "fyearq", "fqtr", "rdq", "sue", "car"]]

,gvkey,permno,fyearq,fqtr,rdq,sue,car
1,001447,59176,2021,1,2021-04-23,0.019271,0.013107


In [13]:
# Check 6: Daily window returns & Notebook values for American Express 2021 Q1
target_row = final_sample[
    (final_sample["gvkey"] == CLIENT_GVKEY) & 
    (final_sample["fyearq"] == 2021) & 
    (final_sample["fqtr"] == 1)
].iloc[0]

day0_val = events[
    (events["gvkey"] == CLIENT_GVKEY) & 
    (events["fyearq"] == 2021) & 
    (events["fqtr"] == 1)
]["day0_tday"].iloc[0]

window_df = daily[
    (daily["permno"] == int(target_row["permno"])) & 
    (daily["tday"].isin([day0_val - 1, day0_val, day0_val + 1]))
].copy().sort_values("date").reset_index(drop=True)

window_df["relative_day"] = window_df["tday"] - day0_val
display_cols = ["permno", "relative_day", "tday", "date", "ret", "vwretd", "ar"]

print("=" * 60)
print(f"Notebook SUE : {target_row['sue']:.6f}")
print(f"Notebook CAR : {target_row['car']:.6f}")
print("=" * 60)
window_df[display_cols]

Notebook SUE : 0.019271
Notebook CAR : 0.013107


,permno,relative_day,tday,date,ret,vwretd,ar
0,59176,-1,98,2021-04-22,-0.000136,-0.007251,0.007115
1,59176,0,99,2021-04-23,-0.019231,0.012026,-0.031257
2,59176,1,100,2021-04-26,0.041641,0.004392,0.037249


In [14]:
# ==============================================================================
# Step 10: Rank by SUE and cut into 3 equal groups (Low, Mid, High)
# ==============================================================================

# 10a. Rank announcements by SUE and divide into 3 equal tertiles
# Note: method="first" breaks ties deterministically to ensure equal groups
final_sample["sue_rank"] = final_sample["sue"].rank(method="first")
final_sample["group"] = pd.qcut(
    final_sample["sue_rank"],
    q=3,
    labels=["Low", "Middle", "High"]
)

# 10b. Compute summary table: number of announcements and average CAR per group
results_table = final_sample.groupby("group", observed=False).agg(
    n_announcements=("car", "count"),
    mean_sue=("sue", "mean"),
    mean_car=("car", "mean")
).reset_index()

# Format for clean presentation
results_table["mean_sue"] = results_table["mean_sue"].round(4)
results_table["mean_car"] = (results_table["mean_car"] * 100).round(2).astype(str) + "%"

print("=" * 65)
print("PROJECT 2: FINAL RESULTS TABLE (SUE TERTILES & AVERAGE CAR)")
print("=" * 65)
results_table

PROJECT 2: FINAL RESULTS TABLE (SUE TERTILES & AVERAGE CAR)


,group,n_announcements,mean_sue,mean_car
0,Low,178,-0.0585,-2.44%
1,Middle,178,0.0002,0.14%
2,High,179,0.0531,3.36%


In [15]:
# Q1: Unique companies per tertile group
final_sample.groupby("group")["gvkey"].nunique()

group
Low       33
Middle    35
High      37
Name: gvkey, dtype: int64

In [16]:
# Q2: Re-run tertile table excluding the 5 largest |CAR| observations
clean_sample = final_sample.drop(top5_car_idx).copy()
clean_sample["group_clean"] = pd.qcut(clean_sample["sue"].rank(method="first"), 3, labels=["Low", "Middle", "High"])
clean_sample.groupby("group_clean")["car"].agg(["count", "mean"])

,count,mean
group_clean,,
Low,177,-0.020574
Middle,176,0.000979
High,177,0.027601


In [17]:
# Q3: Find largest |CAR| announcement for American Express
client = final_sample[final_sample["gvkey"] == CLIENT_GVKEY].copy()
client.loc[client["car"].abs().idxmax(), ["rdq", "fyearq", "fqtr", "sue", "car"]]

rdq       2022-01-25 00:00:00
fyearq                   2021
fqtr                        4
sue                  0.002567
car                  0.115009
Name: 4, dtype: object

In [24]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import wrds

# ==============================================================================
# STEP 0: CREATE DIRECTORY AND CONNECT TO WRDS
# ==============================================================================
# Make sure the figures folder exists for saving graphs
os.makedirs("figures", exist_ok=True)

# ==============================================================================
# STEP 1: LOAD AND FILTER THE CCM LINK TABLE (SIC 37)
# ==============================================================================
# Read the link table
link_raw = pd.read_csv("ccm_link.csv", dtype=str)

# Ensure all column names are lowercase
link_raw.columns = link_raw.columns.str.lower()

# Filter for SIC 37 (Transportation Equipment)
link_37 = link_raw[link_raw["sic2"] == "37"].copy()

# Ensure types for merge keys and dates
link_37["permno"] = link_37["permno"].astype(int)
link_37["linkdt"] = pd.to_datetime(link_37["linkdt"])
link_37["linkenddt"] = pd.to_datetime(link_37["linkenddt"].fillna("2099-12-31"))

# Extract unique identifiers for Compustat and CRSP SQL queries
gvkey_tuple = tuple(link_37["gvkey"].unique().tolist())
permno_tuple = tuple(link_37["permno"].unique().tolist())

print(f"SIC 37: {len(gvkey_tuple)} unique gvkeys, {len(permno_tuple)} unique permnos found.")

# ==============================================================================
# STEP 2: PULL FUNDQ DATA FROM COMPUSTAT (2019 - 2024)
# ==============================================================================
# We pull from 2019 so we have 4 quarters of history to lag for 2021 announcements
fundq_query = f"""
    SELECT gvkey, datadate, fyearq, fqtr, rdq, epspxq, prccq
    FROM comp.fundq
    WHERE gvkey IN {gvkey_tuple}
      AND datadate BETWEEN '2019-01-01' AND '2024-12-31'
      AND indfmt = 'INDL'
      AND datafmt = 'STD'
      AND popsrc = 'D'
      AND consol = 'C'
"""
fundq = db.raw_sql(fundq_query)

# Clean dates and ensure types
fundq["gvkey"] = fundq["gvkey"].astype(str)
fundq["datadate"] = pd.to_datetime(fundq["datadate"])
fundq["rdq"] = pd.to_datetime(fundq["rdq"])
fundq["fyearq"] = fundq["fyearq"].astype(int)
fundq["fqtr"] = fundq["fqtr"].astype(int)

# Merge with the link table to get permno
comp_merged = pd.merge(fundq, link_37, on="gvkey", how="inner")

# Keep only records where announcement date falls inside the valid link window
comp_merged = comp_merged[
    (comp_merged["rdq"] >= comp_merged["linkdt"])
    & (comp_merged["rdq"] <= comp_merged["linkenddt"])
].copy()

# ==============================================================================
# STEP 3: CREATE qtr_id AND SELF-MERGE FOR EPS_LAG4 AND PRC_LAG1 (NO .shift())
# ==============================================================================
# Create a unique chronological integer identifier for each quarter
comp_merged["qtr_id"] = comp_merged["fyearq"] * 4 + comp_merged["fqtr"]

# 3a. Prepare helper table for EPS lag 4 (same quarter last year)
eps_lookup = comp_merged[["gvkey", "qtr_id", "epspxq"]].copy()
eps_lookup["match_qtr_id"] = eps_lookup["qtr_id"] + 4
eps_lookup = eps_lookup.rename(columns={"epspxq": "eps_lag4"})
eps_lookup = eps_lookup[["gvkey", "match_qtr_id", "eps_lag4"]]

# Merge lag 4 EPS back to the main table
comp_merged = pd.merge(
    comp_merged,
    eps_lookup,
    left_on=["gvkey", "qtr_id"],
    right_on=["gvkey", "match_qtr_id"],
    how="left",
)

# 3b. Prepare helper table for Price lag 1 (prior quarter closing price)
prc_lookup = comp_merged[["gvkey", "qtr_id", "prccq"]].copy()
prc_lookup["match_qtr_id_prc"] = prc_lookup["qtr_id"] + 1
prc_lookup = prc_lookup.rename(columns={"prccq": "prc_lag1"})
prc_lookup = prc_lookup[["gvkey", "match_qtr_id_prc", "prc_lag1"]]

# Merge prior quarter price back to the main table
comp_merged = pd.merge(
    comp_merged,
    prc_lookup,
    left_on=["gvkey", "qtr_id"],
    right_on=["gvkey", "match_qtr_id_prc"],
    how="left",
)

# Clean up helper match columns
comp_merged = comp_merged.drop(columns=["match_qtr_id", "match_qtr_id_prc"])

# ==============================================================================
# STEP 4: APPLY RULES 1 TO 5 STEP-BY-STEP
# ==============================================================================
# Rule 1: Announcement date (rdq) must not be missing
step1 = comp_merged[comp_merged["rdq"].notna()].copy()

# Rule 2: Announcement date must be between 2021-01-01 and 2024-12-31
step2 = step1[
    (step1["rdq"] >= "2021-01-01") & (step1["rdq"] <= "2024-12-31")
].copy()

# Rule 3: Valid SUE calculation: SUE = (EPS - EPS_lag4) / Price_lag1
step2["sue"] = (step2["epspxq"] - step2["eps_lag4"]) / step2["prc_lag1"]
step3 = step2[step2["sue"].notna()].copy()

# Rule 4: Price filter (prior quarter price >= $5 to eliminate penny stocks)
step4 = step3[step3["prc_lag1"] >= 5.0].copy()

# Rule 5: Trim extreme SUE outliers (|SUE| <= 1.0)
step5 = step4[step4["sue"].abs() <= 1.0].copy()

# ==============================================================================
# STEP 5: PULL CRSP RETURNS AND MAP DAY 0 (USING TRADING DAY CALENDAR)
# ==============================================================================
# Query daily returns and market value-weighted index
crsp_query = f"""
    SELECT a.permno, a.date, a.ret, b.vwretd
    FROM crsp.dsf a
    LEFT JOIN crsp.dsi b ON a.date = b.date
    WHERE a.permno IN {permno_tuple}
      AND a.date BETWEEN '2020-12-01' AND '2025-01-31'
"""
crsp = db.raw_sql(crsp_query)
crsp["permno"] = crsp["permno"].astype(int)
crsp["date"] = pd.to_datetime(crsp["date"])

# Compute daily abnormal return: ar = ret - vwretd
crsp["ar"] = crsp["ret"] - crsp["vwretd"]

# Create an ordered list of unique trading days
calendar = pd.DataFrame({"trading_date": sorted(crsp["date"].unique())})
calendar["trading_idx"] = np.arange(len(calendar))

# Map rdq to the first trading day on or after rdq (direction='forward')
step5 = step5.sort_values("rdq").reset_index(drop=True)
step5_day0 = pd.merge_asof(
    step5,
    calendar,
    left_on="rdq",
    right_on="trading_date",
    direction="forward",
)
step5_day0 = step5_day0.rename(columns={"trading_date": "day0", "trading_idx": "day0_idx"})

# ==============================================================================
# STEP 6: 3-DAY WINDOW [-1, 0, +1] CAR CALCULATION (NO COMPLEX LOOPS)
# ==============================================================================
# We create 3 simple tables with target trading index offsets: -1, 0, and +1
t_minus_1 = step5_day0.copy()
t_minus_1["target_idx"] = t_minus_1["day0_idx"] - 1
t_minus_1["relative_day"] = -1

t_0 = step5_day0.copy()
t_0["target_idx"] = t_0["day0_idx"]
t_0["relative_day"] = 0

t_plus_1 = step5_day0.copy()
t_plus_1["target_idx"] = t_plus_1["day0_idx"] + 1
t_plus_1["relative_day"] = 1

# Stack the 3 tables together
window_3day = pd.concat([t_minus_1, t_0, t_plus_1], ignore_index=True)

# Merge back with calendar to get the actual date for each trading index
window_3day = pd.merge(
    window_3day,
    calendar,
    left_on="target_idx",
    right_on="trading_idx",
    how="inner",
)
window_3day = window_3day.rename(columns={"trading_date": "event_date"})

# Merge with CRSP daily abnormal returns
window_with_returns = pd.merge(
    window_3day,
    crsp[["permno", "date", "ar"]],
    left_on=["permno", "event_date"],
    right_on=["permno", "date"],
    how="left",
)

# Rule 6: Require all 3 days to have non-missing abnormal returns and sum CAR
summary_car = (
    window_with_returns.groupby(
        ["gvkey", "permno", "fyearq", "fqtr", "rdq", "sue", "epspxq"],
        as_index=False,
    )
    .agg(
        valid_days=("ar", "count"),
        car_decimal=("ar", "sum"),
    )
)

# Keep only announcements with all 3 days present, convert CAR to percentage (%)
final_comp = summary_car[summary_car["valid_days"] == 3].copy()
final_comp["car"] = final_comp["car_decimal"] * 100
final_comp["industry"] = "SIC 37 (Manufacturing)"

# Align existing SIC 61 sample
final_sample_aligned = final_sample.copy()
final_sample_aligned["industry"] = "SIC 61 (Credit/Finance)"

# If CAR in existing sample is in decimals (e.g., 0.05 instead of 5.0%), scale to %
if final_sample_aligned["car"].abs().mean() < 0.5:
    final_sample_aligned["car"] = final_sample_aligned["car"] * 100

# ==============================================================================
# STEP 7: COMBINE SAMPLES (COMMON CORE COLUMNS ONLY) & ASSIGN SUE TERCILES
# ==============================================================================
# 1. Assign industry identifiers
final_comp["industry"] = "SIC 37 (Manufacturing)"
final_sample_aligned = final_sample.copy()
final_sample_aligned["industry"] = "SIC 61 (Credit/Finance)"

# 2. Check CAR unit: if expressed in decimals (e.g., 0.05 instead of 5%), scale to %
if final_sample_aligned["car"].abs().mean() < 0.5:
    final_sample_aligned["car"] = final_sample_aligned["car"] * 100

# 3. Restrict strictly to common columns present in both datasets
core_cols = ["gvkey", "permno", "fyearq", "fqtr", "rdq", "sue", "car", "industry"]

combined = pd.concat(
    [final_sample_aligned[core_cols], final_comp[core_cols]],
    ignore_index=True,
)

# 4. Assign SUE 3-group tercile independently within each industry
combined["sue_group"] = combined.groupby("industry")["sue"].transform(
    lambda s: pd.qcut(s, q=3, labels=["Low", "Middle", "High"])
)

print(f"Combined sample created successfully: Total {len(combined)} observations.")
print(combined["industry"].value_counts())

# ==============================================================================
# STEP 8: FIGURE 1 - SIDE-BY-SIDE BAR CHART WITH 95% CI & N COUNTS
# ==============================================================================
fig1_stats = (
    combined.groupby(["industry", "sue_group"], observed=False)
    .agg(
        mean_car=("car", "mean"),
        std_car=("car", "std"),
        n_obs=("car", "count"),
    )
    .reset_index()
)

# 95% Confidence Interval error bar: 1.96 * (std / sqrt(n))
fig1_stats["ci95"] = 1.96 * (fig1_stats["std_car"] / np.sqrt(fig1_stats["n_obs"]))

fig, ax = plt.subplots(figsize=(8, 5))
x_indices = np.arange(3)
width = 0.35

ind_list = ["SIC 61 (Credit/Finance)", "SIC 37 (Manufacturing)"]
bar_colors = ["#1f77b4", "#ff7f0e"]

for i, ind_name in enumerate(ind_list):
    sub = fig1_stats[fig1_stats["industry"] == ind_name].sort_values("sue_group")
    positions = x_indices + (i - 0.5) * width
    bars = ax.bar(
        positions,
        sub["mean_car"],
        width=width,
        yerr=sub["ci95"],
        capsize=5,
        label=ind_name,
        color=bar_colors[i],
        alpha=0.85,
    )

    # Annotate count N above or below each bar depending on sign
    for bar, n, mean_val, ci_val in zip(bars, sub["n_obs"], sub["mean_car"], sub["ci95"]):
        if mean_val >= 0:
            y_text = mean_val + ci_val + 0.15
            va_align = "bottom"
        else:
            y_text = mean_val - ci_val - 0.35
            va_align = "top"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_text,
            f"N={n}",
            ha="center",
            va=va_align,
            fontsize=9,
            fontweight="bold",
        )

ax.set_xticks(x_indices)
ax.set_xticklabels(["Low SUE", "Middle SUE", "High SUE"], fontsize=11)
ax.set_ylabel("Mean CAR [-1, +1] (%)", fontsize=11)
ax.set_title("Figure 1: Mean 3-Day CAR by SUE Tercile (with 95% CI)", fontsize=12, fontweight="bold")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.legend(frameon=True)
plt.tight_layout()
plt.savefig("figures/figure1.png", dpi=300)
plt.close()
print("Figure 1 saved to figures/figure1.png")

# ==============================================================================
# STEP 9: STATISTICAL TESTS (T-TEST & OLS REGRESSION)
# ==============================================================================
print("\n" + "=" * 65)
print("STATISTICAL RESULTS (T-TEST & REGRESSION)")
print("=" * 65)

for ind_name in ind_list:
    sub = combined[combined["industry"] == ind_name]
    high_car = sub[sub["sue_group"] == "High"]["car"]
    low_car = sub[sub["sue_group"] == "Low"]["car"]

    # Independent Welch's t-test (High vs Low)
    diff = high_car.mean() - low_car.mean()
    t_stat, p_val = stats.ttest_ind(high_car, low_car, equal_var=False)

    # OLS Regression: CAR = a + b * SUE
    model = smf.ols("car ~ sue", data=sub).fit()

    print(f"\n[{ind_name}]")
    print(f"  High CAR Mean: {high_car.mean():.3f}%, Low CAR Mean: {low_car.mean():.3f}%")
    print(f"  Difference (High - Low): {diff:.3f}%")
    print(f"  Welch's t-test: t = {t_stat:.3f}, p-value = {p_val:.4f}")
    print(f"  Regression slope (b): {model.params['sue']:.3f}")
    print(f"  Regression p-value:   {model.pvalues['sue']:.4f}")
    print(f"  Regression R-squared: {model.rsquared:.4f}")

# ==============================================================================
# STEP 10: FIGURE 2 - SCATTER PLOT & REGRESSION LINES
# ==============================================================================
fig, ax = plt.subplots(figsize=(8, 5))

for ind_name, color in zip(ind_list, bar_colors):
    sub = combined[combined["industry"] == ind_name]
    
    # Scatter points
    ax.scatter(sub["sue"], sub["car"], alpha=0.25, s=20, color=color, label=f"{ind_name} data")

    # Fit linear line
    model = smf.ols("car ~ sue", data=sub).fit()
    x_grid = np.linspace(sub["sue"].min(), sub["sue"].max(), 100)
    y_grid = model.params["Intercept"] + model.params["sue"] * x_grid
    
    ax.plot(
        x_grid,
        y_grid,
        color=color,
        linewidth=2.2,
        label=f"{ind_name} fit (b={model.params['sue']:.2f}, R²={model.rsquared:.3f})",
    )

ax.set_xlabel("Standardized Unexpected Earnings (SUE)", fontsize=11)
ax.set_ylabel("Cumulative Abnormal Return CAR [-1, +1] (%)", fontsize=11)
ax.set_title("Figure 2: Earnings Response Coefficients (CAR vs SUE)", fontsize=12, fontweight="bold")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
ax.legend(frameon=True, fontsize=9)
plt.tight_layout()
plt.savefig("figures/figure2.png", dpi=300)
plt.close()
print("Figure 2 saved to figures/figure2.png")

# ==============================================================================
# STEP 11: FIGURE 3 - WHY INDUSTRIES DIFFER (NEGATIVE SUE PROPORTION)
# ==============================================================================
# Define negative earnings surprise (SUE < 0), which is available in both datasets
combined["is_neg_sue"] = (combined["sue"] < 0).astype(int)

neg_sue_df = (
    combined.groupby("industry")["is_neg_sue"]
    .agg(total_count="count", neg_count="sum")
    .reset_index()
)
neg_sue_df["neg_percent"] = (neg_sue_df["neg_count"] / neg_sue_df["total_count"]) * 100

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(
    neg_sue_df["industry"],
    neg_sue_df["neg_percent"],
    color=bar_colors,
    width=0.45,
    alpha=0.85,
)

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + 0.8,
        f"{h:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

ax.set_ylabel("Proportion of Negative SUE (SUE < 0) (%)", fontsize=11)
ax.set_title("Figure 3: Frequency of Negative Earnings Surprises", fontsize=12, fontweight="bold")
ax.set_ylim(0, max(neg_sue_df["neg_percent"]) * 1.25)
plt.tight_layout()
plt.savefig("figures/figure3.png", dpi=300)
plt.close()
print("Figure 3 saved to figures/figure3.png")

# Close connection cleanly
db.close()
print("\nAll tasks completed cleanly.")

SIC 37: 94 unique gvkeys, 84 unique permnos found.
Combined sample created successfully: Total 1493 observations.
industry
SIC 37 (Manufacturing)     958
SIC 61 (Credit/Finance)    535
Name: count, dtype: int64
Figure 1 saved to figures/figure1.png

STATISTICAL RESULTS (T-TEST & REGRESSION)

[SIC 61 (Credit/Finance)]
  High CAR Mean: 3.362%, Low CAR Mean: -2.436%
  Difference (High - Low): 5.798%
  Welch's t-test: t = 6.415, p-value = 0.0000
  Regression slope (b): 21.495
  Regression p-value:   0.0000
  Regression R-squared: 0.0355

[SIC 37 (Manufacturing)]
  High CAR Mean: 2.017%, Low CAR Mean: -1.996%
  Difference (High - Low): 4.013%
  Welch's t-test: t = 5.196, p-value = 0.0000
  Regression slope (b): 8.575
  Regression p-value:   0.0181
  Regression R-squared: 0.0058
Figure 2 saved to figures/figure2.png
Figure 3 saved to figures/figure3.png

All tasks completed cleanly.
